A netwok has 3 components - an input layer, one or more hidden layers and one output layer. Information travels from left to right.

Input layer is not composed of neurons. It just contains raw data and number of nodes = number of features of the data.

Hidden layers are the brain of a neural network. Width of a hidden layer is number of neurons in it and depth is the number of layers. Each neuron in a hidden layer is connected to all outputs of the previous layer. They detect complex patterns like layer 1 may detect shapes, layer 2 may combine shapes to make objects, etc.

Output layer contains number of neurons = number of outputs we want. Activation function used in this is task specific.

# **FROM NEURON TO LAYER**

Say we have n neurons each having m weights. We can then make a weight matrix W of shape m x n where each column vector of the matrix corresponds to the weight vector of that neuron. Similarly we can have a bias vector of size n where each element represents bias of that neuron.

For a single neuron, linear step was wTx where w was weight vector. For this operation it is the matrix vector operation x @ W + b and result is a vector containing pre-activation of all neurons.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Layer(nn.Module):
  def __init__(self, n_input, n_neurons, activation = None):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(n_input, n_neurons)) #Use nn.Parameter(torch.randn()) instead of torch.randn() directly because nn.Parameter tells PyTorch it is a learnable parameter. Only registered parameters are updated by torch.optim.
    self.bias = nn.Parameter(torch.zeros(n_neurons)) #Without nn.Parameters, gradients can still be track upon setting requires_grad = True but we need to pass it in optim everytime.
    self.activation = activation

  def forward(self, x):
    logits = x @ self.weights + self.bias
    if self.activation is not None:
      return self.activation(logits)
    else:
      return logits

In [26]:
n_neurons = 3
n_inputs = 5
batch_size = 2

my_layer = Layer(n_inputs, n_neurons, activation = F.relu)
x = torch.randn(batch_size, n_inputs)
output = my_layer.forward(x)
print(output)

tensor([[0.0000, 0.0000, 0.0000],
        [0.2797, 4.3674, 0.6871]], grad_fn=<ReluBackward0>)


**Practice 1 -** Create an output layer for binary classification problem that should take 16 inputs from a previous hidden layer, using sigmoid as the activation function.

In [16]:
n_inputs = 16
n_neurons = 1 #because output is either yes or no. No other features.

my_layer = Layer(n_inputs, n_neurons, activation = F.sigmoid)
x = torch.randn(n_inputs)
output = my_layer.forward(x)
print(output)


tensor([0.5934], grad_fn=<SigmoidBackward0>)


**Practice 2 -** Create a Layer instance that would be suitable as the output layer for a regression problem (like predicting a house price). It should take 8 inputs.

In [17]:
n_inputs = 8
n_neurons = 1 #We are just interested in the price so one output - that single number.
batch_size = 5

my_layer = Layer(n_inputs, n_neurons) #No activation as the problem is generally Linear Regression. No need for activation as we don't need to deal with boundedness.

x = torch.randn(batch_size, n_inputs)

output = my_layer.forward(x)
print(output)

tensor([[ 0.5994],
        [-1.4978],
        [ 2.1963],
        [-0.0498],
        [ 7.4141]], grad_fn=<AddBackward0>)


**PyTorch built-in Layer Class -** nn.Linear does the same thing as our defined layer.

In [18]:
layer1 = nn.Linear(in_features=8, out_features=1)
x = torch.randn(batch_size, n_inputs)
output = layer1(x)
print(output)

tensor([[-0.0360],
        [ 0.3201],
        [ 0.9455],
        [ 0.1164],
        [ 0.3717]], grad_fn=<AddmmBackward0>)


# **FROM LAYERS TO NETWORKS**

**Goal -** To implement a class for creating a neural network by stacking up layers together.

**To Implement -** A neural network with input layer (2 features), hidden layer 1 (16 neurons, ReLU Activation), hidden layer 2 (8 neurons, ReLU Activation), Output layer (1 neuron, Sigmoid Activation).

In [19]:
class SimpleNetwork(nn.Module):
  def __init__(self, n_inputs, n_hidden1, n_hidden2, n_outputs):
    super().__init__()
    self.hidden1 = Layer(n_inputs, n_hidden1, activation = F.relu)
    self.hidden2 = Layer(n_hidden1, n_hidden2, activation = F.relu)
    self.output = Layer(n_hidden2, n_outputs, activation = F.sigmoid)

    def forward(self, x):
      x = self.hidden1(x)
      x = self.hidden2(x)
      x = self.output(x)
      return x


In [20]:
network = SimpleNetwork(2, 16, 8, 1)
print(network)

SimpleNetwork(
  (hidden1): Layer()
  (hidden2): Layer()
  (output): Layer()
)


In [21]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,1),
    nn.Sigmoid()
)

In [22]:
model

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)

 **Practice 1 -** Implement a deeper network which has a third hidden layer of 4 neurons.

In [23]:
# 1. nn.Sequential implementation

model = nn.Sequential(
    nn.Linear(2,16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,4),
    nn.ReLU(),
    nn.Linear(4,1),
    nn.Sigmoid()
)

print(model)

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
  (7): Sigmoid()
)


In [27]:
#2 Custom Implementation

class DeepNetwork(nn.Module):
  def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, output_size):
    super().__init__()
    self.hidden1 = Layer(input_size, hidden1_size, activation = nn.ReLU)
    self.hidden2 = Layer(hidden1_size, hidden2_size, activation = nn.ReLU)
    self.hidden3 = Layer(hidden2_size, hidden3_size, activation = nn.ReLU)
    self.output = Layer(hidden3_size, output_size, activation = nn.Sigmoid)

  def forward(self, x):
    x = self.hidden1(x)
    x = self.hidden2(x)
    x = self.hidden3(x)
    x = self.output(x)
    return x


my_network = DeepNetwork(2,16,8,4,1)
print(my_network)

DeepNetwork(
  (hidden1): Layer()
  (hidden2): Layer()
  (hidden3): Layer()
  (output): Layer()
)


**Practice 2 -** Define a network for a regression task. It should take 10 input features, have one hidden layer of 32 neurons, and produce a single continuous output. What activation function should you use on the output layer?



In [29]:
class RegressionNetwork(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden = Layer(input_size, hidden_size, activation = nn.ReLU)
    self.output = Layer(hidden_size, output_size) #No activation on output since I don't want to bound the output.

  def forward(self, x):
    x = self.hidden(x)
    x = self.output(x)
    return x